# Stage 2: Dissatisfaction Classifier Training (XLM-RoBERTa, multi-label)

Trains on SentiTaglish_ProductsAndServices_Dissatisfaction_Classification.csv
to classify WHY a negative review is negative, across 13 categories:
def, dam, perf, pmq, lst, var, mi, auth, pss, del, pack, val

Unlike Stage 1 (one sentiment per review), a review can belong to MULTIPLE
dissatisfaction categories at once (72% of the labeled data has 2+ labels).
This means: different loss function (binary cross-entropy per label, not
softmax over one label), different metrics (per-label F1 + micro/macro
averages, not simple accuracy), and a different split strategy (iterative
stratification for multi-label, not sklearn's standard stratify).

**Before running:** set `SMOKE_TEST = True` first, run everything once to
confirm no errors. Then set `SMOKE_TEST = False` and run the full thing.

**Also check:** DATA_PATH matches wherever your uploaded dataset landed
under /kaggle/input/. Use the copy-path icon in the Data panel to get the
exact path rather than typing it by hand.

## 1. Setup

In [1]:
!pip install -q -U transformers datasets scikit-learn
!pip install -q iterative-stratification

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 86.7 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 73.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 78.9 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sklearn-compat 0.1.5 requires scikit-learn<1.9,>=1.2, but you have scikit-learn 1.9.1 which is incompatible.


In [2]:
import re
import unicodedata
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import f1_score, precision_score, recall_score, hamming_loss, accuracy_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from datasets import Dataset

try:
    from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
    HAS_ITERSTRAT = True
except ImportError:
    HAS_ITERSTRAT = False
    print("iterative-stratification not available, will fall back to random split")

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Torch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


## 2. Config

In [3]:
# --- CHECK THIS PATH ---
# Use the copy-path icon in Kaggle's Data panel to get the exact path.
DATA_PATH = "/kaggle/input/datasets/nahokeel/dissatisfactionclassification/SentiTaglish_ProductsAndServices_Dissatisfaction_Classification.csv"

OUTPUT_DIR = "/kaggle/working/stage2_dissatisfaction"

SMOKE_TEST = False
SMOKE_TEST_SIZE = 200
EPOCHS = 10

SEED = 42
MODEL_NAME = "xlm-roberta-base"
MAX_LENGTH = 128
LEARNING_RATE = 2e-5
BATCH_SIZE = 8
PREDICTION_THRESHOLD = 0.5   # sigmoid output above this counts as a positive label

CATEGORY_COLUMNS = ["def", "dam", "perf", "pmq", "lst", "var", "mi",
                    "auth", "pss", "del", "pack", "val"]
NUM_LABELS = len(CATEGORY_COLUMNS)
ID2LABEL = {i: name for i, name in enumerate(CATEGORY_COLUMNS)}
LABEL2ID = {name: i for i, name in enumerate(CATEGORY_COLUMNS)}

torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cpu":
    print("WARNING: no GPU detected. Check Notebook options > Accelerator is set to GPU.")

Using device: cuda


## 3. Text cleaning

In [4]:
SPELLING_MAP = {
    "d2": "dito", "dto": "dito",
    "un": "yun",
    "sya": "siya", "cya": "siya",
    "nde": "hindi", "hnd": "hindi",
    "wla": "wala", "wlang": "walang",
    "eto": "ito",
    "pde": "pwede", "pwd": "pwede",
    "gud": "good",
    "thnx": "thanks", "tnx": "thanks",
    "salamt": "salamat",
}

_REPEATED_CHAR = re.compile(r"(.)\1{2,}")
_WHITESPACE = re.compile(r"\s+")

def clean_text(text: str) -> str:
    if not isinstance(text, str) or not text.strip():
        return ""
    text = unicodedata.normalize("NFKC", text)
    text = text.lower()
    text = _REPEATED_CHAR.sub(r"\1\1", text)
    tokens = text.split()
    tokens = [SPELLING_MAP.get(t, t) for t in tokens]
    text = " ".join(tokens)
    text = _WHITESPACE.sub(" ", text).strip()
    return text

print(clean_text("SOBRAAAA panget ng quality, sira agad!!"))

sobraa panget ng quality, sira agad!!


## 4. Load and prepare data

In [5]:
df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df)} rows")

# Handle either 'review' or 'a' as the text column name (spreadsheet exports
# sometimes rename the first column oddly)
if "review" not in df.columns and "a" in df.columns:
    df = df.rename(columns={"a": "review"})

missing_cols = [c for c in CATEGORY_COLUMNS if c not in df.columns]
if missing_cols:
    raise KeyError(f"Missing expected category columns: {missing_cols}. Found: {list(df.columns)}")

df["review_clean"] = df["review"].apply(clean_text)
df = df[df["review_clean"] != ""].reset_index(drop=True)

labels_matrix = df[CATEGORY_COLUMNS].values.astype(float)
print(f"\nAfter cleaning: {len(df)} rows")

print("\nPer-category positive counts:")
for i, cat in enumerate(CATEGORY_COLUMNS):
    count = int(labels_matrix[:, i].sum())
    print(f"  {cat:6s}: {count:5d} ({100*count/len(df):.1f}%)")

label_counts_per_row = labels_matrix.sum(axis=1)
print(f"\nRows with 2+ labels: {(label_counts_per_row >= 2).sum()} ({100*(label_counts_per_row>=2).sum()/len(df):.1f}%)")

if SMOKE_TEST:
    df = df.sample(min(len(df), SMOKE_TEST_SIZE), random_state=SEED).reset_index(drop=True)
    labels_matrix = df[CATEGORY_COLUMNS].values.astype(float)
    epochs = 1
    print(f"\n[SMOKE TEST] Using {len(df)} rows, 1 epoch")
else:
    epochs = EPOCHS

# Multi-label stratified split: ensures rare categories (e.g. auth at ~2%)
# appear in both train and val, unlike a plain random split.
if HAS_ITERSTRAT and len(df) > 20:
    msss = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
    train_idx, val_idx = next(msss.split(df["review_clean"], labels_matrix))
else:
    from sklearn.model_selection import train_test_split
    train_idx, val_idx = train_test_split(
        np.arange(len(df)), test_size=0.2, random_state=SEED
    )
    print("Using random split (iterstrat unavailable or dataset too small for smoke test)")

train_df = df.iloc[train_idx].reset_index(drop=True)
val_df = df.iloc[val_idx].reset_index(drop=True)
train_labels = labels_matrix[train_idx]
val_labels = labels_matrix[val_idx]

print(f"\nTrain: {len(train_df)} | Val: {len(val_df)}")

Loaded 3408 rows

After cleaning: 3408 rows

Per-category positive counts:
  def   :   710 (20.8%)
  dam   :   502 (14.7%)
  perf  :   392 (11.5%)
  pmq   :   742 (21.8%)
  lst   :   822 (24.1%)
  var   :   878 (25.8%)
  mi    :   379 (11.1%)
  auth  :    61 (1.8%)
  pss   :  1268 (37.2%)
  del   :   224 (6.6%)
  pack  :   251 (7.4%)
  val   :   474 (13.9%)

Rows with 2+ labels: 2460 (72.2%)

Train: 2708 | Val: 700


## 5. Tokenize

In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification",
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

def make_dataset(texts, labels):
    enc = tokenizer(
        list(texts), truncation=True, padding="max_length", max_length=MAX_LENGTH
    )
    enc["labels"] = labels.tolist()
    return Dataset.from_dict(enc)

train_ds = make_dataset(train_df["review_clean"], train_labels)
val_ds = make_dataset(val_df["review_clean"], val_labels)

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## 6. Train

In [7]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))  # sigmoid
    preds = (probs >= PREDICTION_THRESHOLD).astype(int)
    labels = labels.astype(int)

    macro_f1 = f1_score(labels, preds, average="macro", zero_division=0)
    micro_f1 = f1_score(labels, preds, average="micro", zero_division=0)
    macro_precision = precision_score(labels, preds, average="macro", zero_division=0)
    macro_recall = recall_score(labels, preds, average="macro", zero_division=0)
    subset_accuracy = accuracy_score(labels, preds)  # exact match across all 13 labels
    h_loss = hamming_loss(labels, preds)

    return {
        "macro_f1": macro_f1,
        "micro_f1": micro_f1,
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
        "subset_accuracy": subset_accuracy,
        "hamming_loss": h_loss,
    }

common_training_kwargs = dict(
    output_dir=OUTPUT_DIR,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=epochs,
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    logging_steps=20,
    seed=SEED,
    report_to="none",
)
try:
    training_args = TrainingArguments(eval_strategy="epoch", **common_training_kwargs)
except TypeError:
    training_args = TrainingArguments(evaluation_strategy="epoch", **common_training_kwargs)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Macro F1,Micro F1,Macro Precision,Macro Recall,Subset Accuracy,Hamming Loss,Runtime,Samples Per Second,Steps Per Second
1,0.795477,0.751430,0.085611,0.282407,0.114971,0.085432,0.034286,0.147619,7.062400,99.116000,6.230000
2,0.687748,0.628276,0.226766,0.458558,0.326147,0.185021,0.170000,0.119762,7.004300,99.939000,6.282000
3,0.608080,0.570495,0.330067,0.555773,0.476257,0.278162,0.254286,0.108095,7.094900,98.663000,6.202000
4,0.518407,0.520636,0.422092,0.619437,0.511650,0.373789,0.295714,0.099762,7.035700,99.492000,6.254000
5,0.489664,0.490436,0.476891,0.662343,0.593122,0.422257,0.321429,0.092976,7.100600,98.583000,6.197000
6,0.446645,0.458761,0.565685,0.706029,0.736280,0.511189,0.381429,0.084167,7.039900,99.434000,6.250000
7,0.395398,0.435033,0.605409,0.724236,0.731797,0.552750,0.400000,0.080595,7.053800,99.238000,6.238000
8,0.405745,0.426194,0.631492,0.739253,0.739632,0.577740,0.420000,0.077262,7.016500,99.765000,6.271000
9,0.344962,0.418452,0.640356,0.741162,0.748011,0.579179,0.435714,0.075833,7.000100,99.998000,6.286000
10,0.356522,0.418081,0.645564,0.746102,0.736160,0.596514,0.431429,0.075595,6.945900,100.779000,6.335000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1700, training_loss=0.5207391954870785, metrics={'train_runtime': 905.186, 'train_samples_per_second': 29.917, 'train_steps_per_second': 1.878, 'total_flos': 1781421777100800.0, 'train_loss': 0.5207391954870785, 'epoch': 10.0})

## 7. Evaluate and save

In [8]:
metrics = trainer.evaluate()
print("Final validation metrics:")
for k, v in metrics.items():
    print(f"  {k}: {v}")

# Per-category breakdown, since macro/micro averages hide which categories
# are actually struggling (expect 'auth' to be weakest, given only ~2% of data)
predictions = trainer.predict(val_ds)
probs = 1 / (1 + np.exp(-predictions.predictions))
preds = (probs >= PREDICTION_THRESHOLD).astype(int)
val_labels_arr = np.array(val_labels).astype(int)

print("\nPer-category F1:")
for i, cat in enumerate(CATEGORY_COLUMNS):
    f1 = f1_score(val_labels_arr[:, i], preds[:, i], zero_division=0)
    support = int(val_labels_arr[:, i].sum())
    print(f"  {cat:6s}: F1={f1:.3f}  (support={support})")

if not SMOKE_TEST:
    trainer.save_model(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)
    print(f"\nModel saved to {OUTPUT_DIR}")
    print("Download it from the Kaggle 'Output' tab after this session ends.")
else:
    print("\n[SMOKE TEST] Model not saved. Set SMOKE_TEST = False above and re-run for the real training run.")

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Training Loss,Validation Loss,Epoch,Macro F1,Micro F1,Macro Precision,Macro Recall,Subset Accuracy,Hamming Loss,Runtime,Samples Per Second,Steps Per Second
0.356522,0.418081,10,0.645564,0.746102,0.736160,0.596514,0.431429,0.075595,7.219600,96.959000,6.095000


Final validation metrics:
  eval_loss: 0.41808122396469116
  eval_macro_f1: 0.6455637677776208
  eval_micro_f1: 0.7461015593762494
  eval_macro_precision: 0.7361603281796573
  eval_macro_recall: 0.5965144089930265
  eval_subset_accuracy: 0.43142857142857144
  eval_hamming_loss: 0.07559523809523809
  eval_runtime: 7.2196
  eval_samples_per_second: 96.959
  eval_steps_per_second: 6.095


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]



Per-category F1:
  def   : F1=0.752  (support=142)
  dam   : F1=0.774  (support=100)
  perf  : F1=0.554  (support=78)
  pmq   : F1=0.648  (support=148)
  lst   : F1=0.678  (support=164)
  var   : F1=0.875  (support=176)
  mi    : F1=0.623  (support=76)
  auth  : F1=0.000  (support=12)
  pss   : F1=0.851  (support=254)
  del   : F1=0.441  (support=45)
  pack  : F1=0.763  (support=50)
  val   : F1=0.789  (support=95)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model saved to /kaggle/working/stage2_dissatisfaction
Download it from the Kaggle 'Output' tab after this session ends.


In [9]:
# Threshold sweep - paste this as a new cell after "7. Evaluate and save"
# Reuses `predictions` and `val_labels_arr` already computed in that cell

from sklearn.metrics import f1_score

thresholds_to_try = [0.5, 0.4, 0.3, 0.2, 0.1]

print(f"{'Category':8s} " + " ".join(f"t={t:<5.1f}" for t in thresholds_to_try))
for i, cat in enumerate(CATEGORY_COLUMNS):
    scores = []
    for t in thresholds_to_try:
        preds_t = (probs[:, i] >= t).astype(int)
        f1 = f1_score(val_labels_arr[:, i], preds_t, zero_division=0)
        scores.append(f1)
    print(f"{cat:8s} " + " ".join(f"{s:<7.3f}" for s in scores))

Category t=0.5   t=0.4   t=0.3   t=0.2   t=0.1  
def      0.752   0.767   0.765   0.760   0.684  
dam      0.774   0.740   0.735   0.698   0.600  
perf     0.554   0.604   0.671   0.614   0.481  
pmq      0.648   0.662   0.639   0.612   0.547  
lst      0.678   0.721   0.709   0.673   0.569  
var      0.875   0.868   0.857   0.833   0.709  
mi       0.623   0.691   0.678   0.593   0.435  
auth     0.000   0.000   0.000   0.000   0.129  
pss      0.851   0.848   0.851   0.827   0.763  
del      0.441   0.478   0.494   0.530   0.389  
pack     0.763   0.743   0.721   0.612   0.505  
val      0.789   0.804   0.800   0.764   0.643  
